# Exploracao de Dados

Notebook para verificar integridade e estrutura dos dados coletados.

**Fontes disponiveis:**
- **SGS**: Series temporais BCB (Selic, CDI, PTAX, IBC-Br, IGP-M)
- **Expectations**: Expectativas do Relatorio Focus
- **CAGED**: Microdados de emprego formal (MTE)
- **IPEA**: Series agregadas IPEADATA
- **SIDRA**: Series IBGE (IPCA, PIB, etc)
- **Bloomberg**: Dados de mercado (requer terminal)

## 1. Setup

In [ ]:
# Setup do path (notebooks estao em /notebooks, projeto em /)
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

# Nova API unificada - import unico!
import src

# Visualizacao
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

# Query engine para uso direto
qe = src.QueryEngine()

print("Fontes disponiveis:", src.available_sources())

## 2. SGS - Series Temporais BCB

In [ ]:
# Indicadores disponiveis
print("Indicadores SGS:")
for ind in src.sgs.available():
    info = src.sgs.info(ind)
    print(f"  - {ind}: {info['name']} ({info['frequency']})")

In [ ]:
# Status da coleta
src.sgs.get_status()

In [ ]:
# Cobertura temporal - indicadores diarios
print("SGS DAILY - Cobertura Temporal")
print("-" * 70)

for ind in src.sgs.available(frequency='daily'):
    df = src.sgs.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Cobertura temporal - indicadores mensais
print("SGS MONTHLY - Cobertura Temporal")
print("-" * 70)

for ind in src.sgs.available(frequency='monthly'):
    df = src.sgs.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Amostra de dados - Selic
df_selic = src.sgs.read('selic')
print(f"Selic: {len(df_selic):,} registros")
print(f"\nPrimeiros registros:")
display(df_selic.head())
print(f"\nUltimos registros:")
display(df_selic.tail())

In [ ]:
# Visualizacao rapida - Taxas de juros (CDI anualizado)
df_taxas = src.sgs.read('selic', 'cdi', start='2000')

# CDI anualizado (base 252 dias uteis)
df_taxas['cdi_anual'] = (1 + df_taxas['cdi'] / 100) ** 252 - 1
df_taxas['cdi_anual'] = df_taxas['cdi_anual'] * 100

fig, ax = plt.subplots(figsize=(14, 5))
df_taxas['selic'].plot(ax=ax, label='Meta Selic', linewidth=1.5)
df_taxas['cdi_anual'].plot(ax=ax, label='CDI Anualizado (252d)', linewidth=0.8, alpha=0.7)

ax.set_title('Selic vs CDI Anualizado (2000+)')
ax.set_xlabel('Data')
ax.set_ylabel('Taxa (% a.a.)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualizacao rapida - Cambio
df_cambio = src.sgs.read('dolar_ptax', 'euro_ptax', start='2015')

fig, ax = plt.subplots(figsize=(14, 5))
df_cambio['dolar_ptax'].plot(ax=ax, label='Dolar PTAX', linewidth=1)
df_cambio['euro_ptax'].plot(ax=ax, label='Euro PTAX', linewidth=1)

ax.set_title('Taxas de Cambio PTAX (2015+)')
ax.set_xlabel('Data')
ax.set_ylabel('R$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualizacao rapida - IBC-Br
df_ibc = src.sgs.read('ibc_br_bruto', 'ibc_br_dessaz')

fig, ax = plt.subplots(figsize=(14, 5))
df_ibc['ibc_br_bruto'].plot(ax=ax, label='IBC-Br Bruto', linewidth=1, alpha=0.7)
df_ibc['ibc_br_dessaz'].plot(ax=ax, label='IBC-Br Dessazonalizado', linewidth=1.5)

ax.set_title('IBC-Br - Indice de Atividade Economica')
ax.set_xlabel('Data')
ax.set_ylabel('Indice')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Expectations - Relatorio Focus

In [ ]:
# Indicadores disponiveis
print("Indicadores Expectations:")
for ind in src.expectations.available():
    info = src.expectations.info(ind)
    print(f"  - {ind}: {info.get('indicator', 'N/A')} ({info.get('endpoint', 'N/A')})")

In [ ]:
# Status da coleta
src.expectations.get_status()

In [ ]:
# Cobertura temporal por indicador
print("EXPECTATIONS - Cobertura Temporal")
print("-" * 70)

for ind in src.expectations.available():
    df = src.expectations.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>8,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

In [ ]:
# Amostra de dados - IPCA Anual
df_ipca = src.expectations.read('ipca_anual')
print(f"IPCA Anual: {len(df_ipca):,} registros")
print(f"Colunas: {list(df_ipca.columns)}")
print(f"\nAmostra:")
display(df_ipca.sample(5))

In [ ]:
# Visualizacao rapida - Mediana das expectativas de IPCA
if 'Mediana' in df_ipca.columns:
    mediana_diaria = df_ipca.groupby(df_ipca.index)['Mediana'].mean()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    mediana_diaria.plot(ax=ax, linewidth=0.5)
    
    ax.set_title('IPCA Anual - Mediana das Expectativas')
    ax.set_xlabel('Data da Pesquisa')
    ax.set_ylabel('Expectativa (%)')
    plt.tight_layout()
    plt.show()

## 4. IPEA - Series Agregadas

In [ ]:
# Indicadores disponiveis
print("Indicadores IPEA:")
for ind in src.ipea.available():
    info = src.ipea.info(ind)
    print(f"  - {ind}: {info['name']}")

In [ ]:
# Status da coleta
src.ipea.get_status()

In [ ]:
# Cobertura temporal
print("IPEA - Cobertura Temporal")
print("-" * 70)

for ind in src.ipea.available():
    df = src.ipea.read(ind)
    if not df.empty:
        print(f"{ind:25} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>6,} registros")
    else:
        print(f"{ind:25} | SEM DADOS")

In [ ]:
# Amostra de dados - Saldo CAGED
df_saldo = src.ipea.read('caged_saldo')
print(f"Saldo CAGED: {len(df_saldo):,} registros")
print(f"\nUltimos registros:")
display(df_saldo.tail())

In [ ]:
# Visualizacao rapida - Saldo CAGED
fig, ax = plt.subplots(figsize=(14, 5))

df_saldo['value'].plot(ax=ax, linewidth=1, color='steelblue')
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

ax.set_title('Saldo de Empregos Formais (CAGED)')
ax.set_xlabel('Data')
ax.set_ylabel('Saldo (pessoas)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

## 5. CAGED - Microdados

In [ ]:
# Datasets disponiveis
print("Datasets CAGED:")
for ds, info in src.caged.info().items():
    print(f"  - {ds}: {info['name']}")

In [ ]:
# Periodos disponiveis
periods = src.caged.available_periods()
if periods:
    print(f"Periodos disponiveis: {periods[0]} a {periods[-1]}")
    print(f"Total: {len(periods)} meses")
else:
    print("Nenhum dado coletado ainda. Execute: src.caged.collect()")

In [ ]:
# Amostra de dados - CAGED 2025-10 (se disponivel)
try:
    df_caged = src.caged.read(year=2025, month=10)
    print(f"CAGED 2025-10: {len(df_caged):,} registros")
    print(f"Colunas: {list(df_caged.columns)}")
    print(f"\nResumo:")
    print(f"  Admissoes: {(df_caged['saldomovimentacao'] == 1).sum():,}")
    print(f"  Desligamentos: {(df_caged['saldomovimentacao'] == -1).sum():,}")
    print(f"  Saldo: {df_caged['saldomovimentacao'].sum():,}")
    print(f"\nAmostra:")
    display(df_caged.sample(5))
except Exception as e:
    print(f"Erro ao carregar dados: {e}")

In [ ]:
# Saldo por UF (usando metodo de agregacao)
try:
    df_saldo_uf = src.caged.saldo_por_uf(year=2025)
    print("Saldo CAGED por UF - 2025:")
    display(df_saldo_uf.head(10))
except Exception as e:
    print(f"Erro: {e}")

In [ ]:
# Saldo mensal (usando metodo de agregacao)
try:
    df_saldo_mes = src.caged.saldo_mensal(year=2025)
    print("Saldo CAGED Mensal - 2025:")
    display(df_saldo_mes)
    
    # Visualizacao
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['green' if x > 0 else 'red' for x in df_saldo_mes['saldo']]
    ax.bar(df_saldo_mes['mes'], df_saldo_mes['saldo'], color=colors, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_title('Saldo CAGED por Mes - 2025')
    ax.set_xlabel('Mes')
    ax.set_ylabel('Saldo')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Erro: {e}")

## 6. SIDRA - Series IBGE

In [ ]:
# Indicadores disponiveis
print("Indicadores SIDRA:")
for ind in src.sidra.available():
    info = src.sidra.info(ind)
    print(f"  - {ind}: {info.get('name', 'N/A')}")

In [ ]:
# Status da coleta
src.sidra.get_status()

In [ ]:
# Cobertura temporal
print("SIDRA - Cobertura Temporal")
print("-" * 70)

for ind in src.sidra.available():
    df = src.sidra.read(ind)
    if not df.empty:
        print(f"{ind:20} | {df.index.min().strftime('%Y-%m-%d')} a {df.index.max().strftime('%Y-%m-%d')} | {len(df):>6,} registros")
    else:
        print(f"{ind:20} | SEM DADOS")

## 7. Bloomberg (se disponivel)

In [ ]:
# Indicadores disponiveis
print("Indicadores Bloomberg:")
for ind in src.bloomberg.available():
    info = src.bloomberg.info(ind)
    print(f"  - {ind}: {info.get('ticker', 'N/A')} ({info.get('category', 'N/A')})")

In [ ]:
# Verificar se tem dados coletados
try:
    df_ibov = src.bloomberg.read('ibov_points')
    if not df_ibov.empty:
        print(f"Ibovespa: {len(df_ibov):,} registros")
        print(f"Periodo: {df_ibov.index.min().strftime('%Y-%m-%d')} a {df_ibov.index.max().strftime('%Y-%m-%d')}")
        display(df_ibov.tail())
    else:
        print("Nenhum dado coletado. Requer Bloomberg Terminal.")
except Exception as e:
    print(f"Bloomberg nao disponivel: {e}")

## 8. QueryEngine - SQL Direto

Para queries mais complexas, use o QueryEngine com SQL direto via DuckDB.

In [ ]:
# Exemplo: Media anual da Selic
df_media_selic = qe.sql("""
    SELECT 
        strftime(date, '%Y') as ano,
        AVG(value) as media_selic,
        MIN(value) as min_selic,
        MAX(value) as max_selic
    FROM '{raw}/bacen/sgs/daily/selic.parquet'
    GROUP BY ano
    ORDER BY ano
""")

print("Media Anual da Selic:")
display(df_media_selic.tail(10))

In [ ]:
# Exemplo: Agregacao CAGED com SQL
try:
    df_setor = qe.sql("""
        SELECT 
            secao as setor,
            SUM(saldomovimentacao) as saldo,
            COUNT(*) as registros,
            AVG(CAST(REPLACE(salario, ',', '.') AS DOUBLE)) as salario_medio
        FROM '{raw}/mte/caged/cagedmov_2025-*.parquet'
        GROUP BY secao
        ORDER BY saldo DESC
    """)
    
    print("Saldo por Setor - CAGED 2025:")
    display(df_setor.head(10))
except Exception as e:
    print(f"Erro: {e}")